### Human-in-the-Loop (HITL)

**Human-in-the-Loop (HITL)** is a mechanism that allows a human to **review and control an AI agent's actions before they are executed**. In LangChain/LangGraph, HITL can interrupt an agent when it reaches a sensitive tool call, such as sending an email, making a payment, deleting data, or updating a database. The agent's state is preserved using a **checkpointer**, and execution can then be resumed using a `Command` with a decision such as **approve** or **reject**. This provides an additional layer of **safety, control, and human oversight** for actions that should not be performed automatically.

| Component        | Purpose                                                           |
| ---------------- | ----------------------------------------------------------------- |
| **Interrupt**    | Pauses agent execution before a sensitive action                  |
| **Human Review** | Allows a person to inspect the proposed action                    |
| **Approve**      | Allows the agent to continue and execute the action               |
| **Reject**       | Prevents the action from being executed                           |
| **`Command`**    | Resumes the interrupted agent with the human's decision           |
| **Checkpointer** | Saves the agent state while execution is interrupted              |
| **`thread_id`**  | Identifies the conversation/workflow whose state is being resumed |

**Typical flow:**

```text
User Request
     ↓
AI Agent
     ↓
Tool Call
     ↓
Interrupt
     ↓
Human Review
   ↙       ↘
Approve   Reject
   ↓         ↓
Execute    Stop/Modify
```

**Key idea:** HITL is used when **AI autonomy should be combined with human judgment**, especially for actions that are sensitive, irreversible, or have real-world consequences.


In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.agents.middleware import HumanInTheLoopMiddleware

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


# --------------------------------------------------
# TOOLS
# --------------------------------------------------

@tool
def read_email(email_id: str) -> str:
    """Read an email using its email ID."""
    emails = {
        "101": {
            "from": "hr@company.com",
            "subject": "Interview Invitation",
            "body": "Your interview is scheduled for Monday at 10 AM."
        },
        "102": {
            "from": "manager@company.com",
            "subject": "Project Update",
            "body": "Please send the project status report today."
        }
    }

    email = emails.get(email_id)

    if not email:
        return "Email not found."

    return (
        f"From: {email['from']}\n"
        f"Subject: {email['subject']}\n"
        f"Body: {email['body']}"
    )


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return (
        f"Email successfully sent.\n"
        f"To: {to}\n"
        f"Subject: {subject}\n"
        f"Body: {body}"
    )


# --------------------------------------------------
# HUMAN-IN-THE-LOOP MIDDLEWARE
# --------------------------------------------------

hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": True,
        "read_email": False
    }
)


# --------------------------------------------------
# CHECKPOINTER
# --------------------------------------------------

checkpointer = InMemorySaver()


# --------------------------------------------------
# CREATE AGENT
# --------------------------------------------------

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    middleware=[hitl_middleware],
    checkpointer=checkpointer
)


# --------------------------------------------------
# CONFIG + THREAD ID
# --------------------------------------------------

config = {
    "configurable": {
        "thread_id": "email_thread_001"
    }
}


# --------------------------------------------------
# ASK AGENT TO SEND EMAIL
# --------------------------------------------------

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
                Send an email to manager@company.com.
                Subject: Project Update
                Body: The project is progressing well and will be
                completed on schedule.
                """
            }
        ]
    },
    config=config
)


# --------------------------------------------------
# RESULT BEFORE HUMAN DECISION
# --------------------------------------------------

print("\n========== INTERRUPT ==========")

print(response)


# --------------------------------------------------
# APPROVE / REJECT
# --------------------------------------------------

# APPROVE:
# command = Command(
#     resume={
#         "decisions": [
#             {
#                 "type": "approve"
#             }
#         ]
#     }
# )


# REJECT:
command = Command(
    resume={
        "decisions": [
            {
                "type": "reject",
                "message": "Do not send this email."
            }
        ]
    }
)


# --------------------------------------------------
# RESUME AGENT
# --------------------------------------------------

result = agent.invoke(
    command,
    config=config
)


# --------------------------------------------------
# SHOW RESULT
# --------------------------------------------------

print("\n========== FINAL RESULT ==========")

for message in result["messages"]:
    print(f"\n{type(message).__name__}:")
    print(message.content)

d:\langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



========== INTERRUPT ==========
{'messages': [HumanMessage(content='\n                Send an email to manager@company.com.\n                Subject: Project Update\n                Body: The project is progressing well and will be\n                completed on schedule.\n                ', additional_kwargs={}, response_metadata={}, id='e16c7b35-c714-4e6c-b267-9e47972976ea'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email. We have a function send_email. We need to call it with body, subject, to. The body is "The project is progressing well and will be completed on schedule." The subject is "Project Update". The to is "manager@company.com". Let\'s do that.', 'tool_calls': [{'id': 'fc_1b5d14e8-c4a8-492c-b15f-9794962b3889', 'function': {'arguments': '{"body":"The project is progressing well and will be completed on schedule.","subject":"Project Update","to":"manager@company.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_met